# Genotype–Phenotype Heterogeneity and Infertility Management Dataset Exploration with `mlcroissant`
This notebook provides a guided exploration of the FAIR² clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic information
print(f"{metadata.name}: {metadata.description}\nPublished: {metadata.datePublished}\nKeywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

The Croissant schema organizes tabular data into multiple record sets, each with fields and columns referenced by their unique `@id`. Here we enumerate the available record sets and their fields.

In [ ]:
# List record sets, field @ids, and columns
record_sets = dataset.metadata.recordSet
print(f"Number of record sets: {len(record_sets)}")

overview = []
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    if 'field' in rs:
        print("  Fields:")
        for fld in rs['field']:
            print(f"   - @id: {fld['@id']} Name: {fld.get('name','')} DataType: {fld.get('dataType','')}")
        overview.append({'record_set_id': rs['@id'], 'fields': [fld['@id'] for fld in rs['field']]})
    elif 'column' in rs:  # Sometimes columns used for tabular data
        print("  Columns:")
        for col in rs['column']:
            print(f"   - @id: {col['@id']} Name: {col.get('name','')} DataType: {col.get('dataType','')}")
        overview.append({'record_set_id': rs['@id'], 'columns': [col['@id'] for col in rs['column']]})


## 3. Data Extraction
Extract and load data from each record set into DataFrames for further analysis.

We use the record set and field/column `@id` values from the overview to load each record set's data.

In [ ]:
# Prepare a list of available record set @ids
record_set_ids = [entry['record_set_id'] for entry in overview]
dataframes = {}
for rs_id in record_set_ids:
    # Load records for each record set using its @id
    recs = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(recs)

# Show available columns for the first record set
first_rs_id = record_set_ids[0] if record_set_ids else None
if first_rs_id:
    print(f"Columns for record set {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's choose a numeric field (such as age, hormone value, or measurement) from a record set for demonstration. We reference them via their `@id` field.

In [ ]:
# Example: Use Table 2 (hormone levels, imaging)
# Suppose Table 2's record set @id is 'https://api.app.sen.science/frontiers/7810165/table2'
record_set_id = record_set_ids[1] if len(record_set_ids) > 1 else first_rs_id
# Let's pick a numeric column, e.g., 'cr:Field_hormone_level' (replace with actual @id as listed previously!)
numeric_field_id = None
if record_set_id and not dataframes[record_set_id].empty:
    # Choose first numeric-looking column
    for col in dataframes[record_set_id].columns:
        if any(keyword in col.lower() for keyword in ['age', 'level', 'value', 'concentration']):
            numeric_field_id = col
            break

if numeric_field_id:
    # Set a threshold for filtering (example: hormone level > 10)
    threshold = 10
    filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n", filtered_df.head())

    # Normalize the numeric field
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field, e.g., 'cr:Field_adrenal_CT_findings' (replace with actual @id)
    group_field_id = None
    for col in dataframes[record_set_id].columns:
        if any(keyword in col.lower() for keyword in ['ct', 'ultrasound', 'finding', 'type', 'phenotype']):
            group_field_id = col
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions and relationships. For example, plot distribution of a numeric value across groups.


In [ ]:
# Plot numeric field distribution, colored/grouped by phenotype or findings
if numeric_field_id and group_field_id:
    plt.figure(figsize=(7,5))
    for grp, grp_df in filtered_df.groupby(group_field_id):
        plt.hist(grp_df[numeric_field_id], bins=5, alpha=0.6, label=str(grp))
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field_id} grouped by {group_field_id}')
    plt.legend()
    plt.show()
else:
    print("Insufficient data to visualize.")

## 6. Conclusion
This notebook used the `mlcroissant` library to load, explore, and process the FAIR² clinical dataset. Key findings include:
- Structure and fields referenced by `@id` for reproducibility.
- Filtered and normalized numeric measures for targeted clinical analysis.
- Grouped visualizations by phenotypic or imaging findings.

Further exploration may involve linking genetic (Table 3) and clinical/hormone data for genotype-phenotype correlation, or adapting processing steps to your research needs.